In [ ]:
# Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#     https://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Module 07 — Computer Use

With computer use, **`claude-opus-4-8`** only **emits actions** (screenshot, click, type, key, scroll) — **you execute each one against a real screen** and return the result (usually a screenshot). Because it needs a real display, a plain notebook **cannot fully run it**.

So this module comes in **two tiers**:

1. **This notebook** teaches the tool definition and the **action-loop shape** (with a mock screen).
2. **`04-demos/claude-computer-use-env/`** runs it for real in a sandboxed environment with a virtual display.

Computer use is a **BETA tool**, invoked via **`client.beta.messages.create`**. This builds on **Modules 00–06** and uses the same **`AnthropicVertex` / ADC** path.

## Part B — Bootstrap

Setup mirrors Module 00 — see that module for full detail (prerequisites, ADC, logging).

In [ ]:
%pip install --quiet "anthropic[vertex]" pillow

In [ ]:
# ===== CONDENSED BOOTSTRAP (mirrors Module 00) =====
import subprocess
from anthropic import AnthropicVertex

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
LOCATION   = "global"
MODEL      = "claude-opus-4-8"

assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or edit PROJECT_ID directly."
)

client = AnthropicVertex(project_id=PROJECT_ID, region=LOCATION)
print(f"✅ Client ready (project={PROJECT_ID}, region={LOCATION}, model={MODEL}).")

## Part C — Define the computer tool

The computer tool is **Anthropic-defined** (you don't write its schema) and requires the **beta** flag:

```python
COMPUTER_TOOL = {"type": "computer_20251124", "name": "computer",
                 "display_width_px": 1024, "display_height_px": 768, "display_number": 1}
BETA = "computer-use-2025-11-24"
```

Call it through **`client.beta.messages.create(..., betas=[BETA])`**. It's often paired with the **`bash`** and **`text_editor`** tools for full control of a machine.

In [ ]:
COMPUTER_TOOL = {
    "type": "computer_20251124",
    "name": "computer",
    "display_width_px": 1024,
    "display_height_px": 768,
    "display_number": 1,
}
BETA = "computer-use-2025-11-24"

try:
    resp = client.beta.messages.create(
        model=MODEL,
        max_tokens=1024,
        tools=[COMPUTER_TOOL],
        betas=[BETA],
        messages=[{"role": "user", "content": "Open a web browser and go to example.com."}],
    )
    print(f"stop_reason: {getattr(resp, 'stop_reason', None)}")
    for block in resp.content:
        if getattr(block, "type", None) == "tool_use":
            print(f"  action name : {getattr(block, 'name', None)}")
            print(f"  action input: {getattr(block, 'input', None)}")
except Exception as e:
    print(f"⚠️  Call failed: {type(e).__name__}: {e}")
    print("   Computer use may need enabling, or availability confirmed for this model/region.")

## Part D — The action loop (shape) with a MOCK screenshot

The loop is: the model returns a **`tool_use`** action → **you perform it on a real screen** → you return a **`tool_result`** whose content is a **screenshot image block** → repeat until the model stops requesting actions.

A screenshot `tool_result` is an **image** block:
```python
{"type": "tool_result", "tool_use_id": <id>,
 "content": [{"type": "image",
              "source": {"type": "base64", "media_type": "image/png", "data": <b64>}}]}
```

Below is **one illustrative round** using a **mock blank screen** — purely to show the round-trip shape.

In [ ]:
import base64
import io
from PIL import Image

# Generate a MOCK blank 1024x768 white screenshot (self-contained — NOT a real screen).
buf = io.BytesIO()
Image.new("RGB", (1024, 768), "white").save(buf, format="PNG")
mock_screenshot_b64 = base64.standard_b64encode(buf.getvalue()).decode()

# Respond to EVERY tool_use block from Part C with its own screenshot tool_result.
tool_uses = [b for b in resp.content if getattr(b, "type", None) == "tool_use"]

if tool_uses:
    tool_results = [{
        "type": "tool_result",
        "tool_use_id": block.id,
        "content": [{
            "type": "image",
            "source": {"type": "base64", "media_type": "image/png", "data": mock_screenshot_b64},
        }],
    } for block in tool_uses]

    messages = [
        {"role": "user", "content": "Open a web browser and go to example.com."},
        {"role": "assistant", "content": resp.content},          # the model's requested action(s)
        {"role": "user", "content": tool_results},               # one result per tool_use block
    ]
    resp2 = client.beta.messages.create(
        model=MODEL,
        max_tokens=1024,
        tools=[COMPUTER_TOOL],
        betas=[BETA],
        messages=messages,
    )
    print(f"next stop_reason: {getattr(resp2, 'stop_reason', None)}")
    for block in resp2.content:
        if getattr(block, "type", None) == "tool_use":
            print(f"  next action : {getattr(block, 'name', None)} -> {getattr(block, 'input', None)}")
        elif getattr(block, "type", None) == "text":
            print(f"  text: {block.text}")
else:
    print("No tool_use action from Part C to respond to (see the note/error above).")

print("\nℹ️  The blank screen is ONLY to show the round-trip shape.")
print("   Real use requires a real display — see Tier 2 (04-demos/claude-computer-use-env/).")

## Part E — Tier 2 pointer + safety + notes

**Tier 2 — the real environment.** Computer use runs against a real display, so the demo in **`04-demos/claude-computer-use-env/`** runs in a **sandboxed container** with a **virtual display (Xvfb)** plus the agent loop that executes actions and returns screenshots.

> ⚠️ **SAFETY.** Run computer use **only in an isolated sandbox/VM, never on your own machine**. Get **explicit user consent** before letting the model act. Screen content the model reads can carry **prompt-injection** risk — treat anything on screen as untrusted input.

### Notes

- **Beta tool:** `computer_20251124` with beta flag `"computer-use-2025-11-24"`, called via `client.beta.messages.create`.
- **Display size:** keep it modest (≤ ~1280×800) — oversized displays hurt accuracy and cost more tokens.
- **Pair with `bash` / `text_editor`** for full machine control beyond pointer/keyboard.
- Confirm availability on the **Opus 4.8 Model Garden card** and the current tool version in the docs: https://docs.claude.com/en/docs/agents-and-tools/tool-use/computer-use-tool

## Part F — Recap

- Computer use is a **beta** tool: the model **emits actions**, **you execute them** on a real screen and return a screenshot **`tool_result`** (image block).
- Define it with `{"type": "computer_20251124", "name": "computer", ...}` and call `client.beta.messages.create(..., betas=["computer-use-2025-11-24"])`.
- The **action loop** repeats action → execute → screenshot until the model stops.
- A notebook can teach the **shape**; real runs need a **sandboxed display** — see **`04-demos/claude-computer-use-env/`**.
- **Safety first:** isolated sandbox, user consent, prompt-injection awareness.

**Next: Module 08 — Memory tool.**